# 流式输出 Streaming

流式输出让 AI 的回复像打字一样逐字显示。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

print("导入完成")

## stream_mode="messages"——逐 Token 流式

最细粒度模式，每个 chunk 对应一个 Token。

In [ ]:
model = init_chat_model("deepseek:deepseek-v4-flash")
agent = create_agent(model=model, system_prompt="你是菜鸟教程 RUNOOB 的助手。")

print("实时流式输出：")
for msg_chunk, metadata in agent.stream({"messages": [HumanMessage(content="用一句话介绍菜鸟教程 RUNOOB")]}, stream_mode="messages"):
    if msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

### 理解 metadata

metadata 包含 chunk 的来源信息。

In [ ]:
print("查看 metadata 信息：\n")
for msg_chunk, metadata in agent.stream({"messages": [HumanMessage(content="你好")]}, stream_mode="messages"):
    if msg_chunk.content and len(msg_chunk.content) > 5:
        print(f"内容: {msg_chunk.content}")
        print(f"来源节点: {metadata.get('langgraph_node')}")
        break

## stream_mode="updates"——逐步查看 Agent 执行

构建显示"思考过程"的界面。

In [ ]:
from langchain.tools import tool

@tool
def search_course(keyword: str) -> str:
    courses = {"python": "Python3 基础教程（30章，20小时）"}
    return courses.get(keyword.lower(), "未找到")

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0), tools=[search_course], system_prompt="你是课程顾问。")

print("=== Agent 执行过程 ===\n")
for chunk in agent.stream({"messages": [HumanMessage(content="查一下 Python 课程")]}, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"[{node_name}]", end=" ")
        if "messages" in update:
            for msg in update["messages"]:
                if msg.type == "ai" and hasattr(msg, 'tool_calls') and msg.tool_calls:
                    print(f"请求调用: {[tc['name'] for tc in msg.tool_calls]}")
                elif msg.type == "tool":
                    print(f"工具返回 [{msg.name}]: {msg.content}")
                elif msg.type == "ai" and msg.content:
                    print(f"回复: {msg.content[:80]}")

## stream_mode="custom"——自定义事件

Middleware 通过 `runtime.stream_writer()` 发送。

In [ ]:
from langchain.agents.middleware import before_model, after_model

@before_model
def notify_before(state, runtime):
    runtime.stream_writer({"type": "status", "message": "正在思考..."})
    return None

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0), tools=[search_course], middleware=[notify_before], system_prompt="你是课程顾问。")

for mode, chunk in agent.stream({"messages": [HumanMessage(content="查一下 Python 课程")]}, stream_mode=["updates", "custom"]):
    if mode == "custom":
        print(f"[事件] {chunk['message']}")

## 异步流式输出

Web 服务中使用 `astream()`。

In [ ]:
import asyncio

async def stream_agent():
    agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash"), system_prompt="你是助手。")
    full = ""
    async for msg_chunk, metadata in agent.astream({"messages": [HumanMessage(content="一句话介绍菜鸟教程")]}, stream_mode="messages"):
        if msg_chunk.content:
            full += msg_chunk.content
            print(msg_chunk.content, end="", flush=True)
    print(f"\n\n长度: {len(full)} 字")

# await stream_agent()
print("取消注释 await stream_agent() 运行")

**术语：** AIMessageChunk、stream_mode、stream_writer()、astream()